<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/03-jdbc-read-write.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 5.3 — JDBC read/write

Live demo against **Derby** (embedded database whose driver ships inside
pyspark — no setup). Same API as Postgres/MySQL; only the URL and driver
coordinates change. Derby files land in `output/`; cleaned up at the end.

If the Derby cells error on your setup (driver not on the classpath in
some environments), the code is still the exact production pattern —
swap in any database + `--packages` and it runs.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("5.3-jdbc")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

URL = "jdbc:derby:output/demo_db;create=true"
DRIVER = "org.apache.derby.jdbc.EmbeddedDriver"

## Write a table (note the coalesce — connection etiquette)

In [ ]:
payments = spark.range(1, 10_001).select(
    col("id").alias("payment_id"),
    (col("id") % 200).alias("customer_id"),
    F.round(F.rand(seed=1) * 500, 2).alias("amount"),
)

(payments
 .coalesce(2)                          # cap concurrent connections
 .write.format("jdbc")
 .option("url", URL)
 .option("driver", DRIVER)
 .option("dbtable", "payments")
 .option("batchsize", 5000)
 .mode("overwrite")
 .save())
print("written")

## Read #1: the unpartitioned default — ONE task does everything

In [ ]:
serial = (
    spark.read.format("jdbc")
    .option("url", URL).option("driver", DRIVER)
    .option("dbtable", "payments")
    .load()
)
print("partitions:", serial.rdd.getNumPartitions())   # 1 — single connection, single thread
print("rows:      ", serial.count())

## Read #2: partitioned — bounds from a real MIN/MAX pre-query

In [ ]:
# bounds come from the data, not from hope:
bounds = (
    spark.read.format("jdbc")
    .option("url", URL).option("driver", DRIVER)
    .option("dbtable", "(SELECT MIN(payment_id) lo, MAX(payment_id) hi FROM payments) t")
    .load()
).first()
print("bounds:", bounds.LO, "->", bounds.HI)

parallel = (
    spark.read.format("jdbc")
    .option("url", URL).option("driver", DRIVER)
    .option("dbtable", "payments")
    .option("partitionColumn", "payment_id")
    .option("lowerBound", int(bounds.LO))
    .option("upperBound", int(bounds.HI) + 1)
    .option("numPartitions", 4)
    .option("fetchsize", 5000)
    .load()
)
print("partitions:", parallel.rdd.getNumPartitions())
print("rows/partition:", parallel.rdd.glom().map(len).collect())   # even split

## 'Bounds don't filter' — and skew from wrong bounds, demonstrated

In [ ]:
bad_bounds = (
    spark.read.format("jdbc")
    .option("url", URL).option("driver", DRIVER)
    .option("dbtable", "payments")
    .option("partitionColumn", "payment_id")
    .option("lowerBound", 1)
    .option("upperBound", 1000)          # table actually goes to 10,000
    .option("numPartitions", 4)
    .load()
)
print("total rows (nothing lost):", bad_bounds.count())
print("rows/partition:", bad_bounds.rdd.glom().map(len).collect())
# last partition ate rows 750..10000 — one task does 92% of the work

## Pushdown into the generated SQL — and the subquery-as-dbtable pattern

In [ ]:
serial.filter(col("amount") > 400).explain()   # PushedFilters: [GreaterThan(amount,400.0)]

# make the DATABASE do the selection before bytes move:
big_only = (
    spark.read.format("jdbc")
    .option("url", URL).option("driver", DRIVER)
    .option("dbtable", "(SELECT payment_id, amount FROM payments WHERE amount > 400) t")
    .load()
)
print("rows after DB-side filter:", big_only.count())

## Cleanup

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)
print("cleaned up. (Derby also drops a derby.log in the notebook dir — deletable.)")
import pathlib
pathlib.Path("derby.log").unlink(missing_ok=True)

## Exercises

1. Re-read `payments` partitioned on `customer_id` (0–199) with 4 partitions. Compare `glom` sizes with the `payment_id` version — why still even here, and what data shape would break it?
2. Use the `predicates=` list form of `spark.read.jdbc()` to split the read into amount bands: <100, 100–300, >300. Check `getNumPartitions()`.
3. Append 100 more rows with `mode("append")`, then run the whole write again with `mode("overwrite")` — what's the final count, and what did overwrite do to the table in between?
4. Sketch (code, no run) the staging-swap pattern for a nightly load into `analytics.daily_totals` — which parts are Spark, which are database SQL?

In [ ]:
# your exercise code here
